# 🧠 CalRetail — Inventory Health Monitoring
## Goal
Assign SKU status indicators (Healthy, At Risk, Critical) from a genuine composite of stock
coverage, overstock, and real supplier reliability.

## Algorithmic Explanation
**Multi-Metric Composite Inventory Scoring**
1. Calculate daily purchase consumption velocity rate from real transactions.
2. Compute days cover (current stock / daily velocity) and map to stockout risk via an inverse
   sigmoid.
3. Blend stockout risk, overstock flag, and the product's real supplier reliability score using
   PCA-derived weights (`adaptive_thresholds.get_inventory_health_weights`) into one composite
   health score — previously the "composite" score used only stockout risk and never touched
   overstock or supplier reliability at all, despite loading the suppliers table.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data' / 'processed'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
from backend.utils.adaptive_thresholds import get_inventory_health_weights

inv = pd.read_csv(processed_dir / 'inventory.csv')
tx = pd.read_csv(processed_dir / 'transactions.csv')
suppliers = pd.read_csv(processed_dir / 'suppliers.csv')
prod = pd.read_csv(processed_dir / 'products.csv')

# Compute daily velocity over 30 days
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'])
max_date = tx['transaction_date'].max()
recent_tx = tx[tx['transaction_date'] >= (max_date - pd.Timedelta(days=30))]

velocity = recent_tx.groupby('product_id')['quantity'].sum().reset_index()
velocity['daily_velocity'] = velocity['quantity'] / 30.0

# Real supplier reliability per product (previously loaded but never used).
product_supplier_map = dict(zip(prod['product_id'], prod['supplier_id']))
supplier_reliability_map = dict(zip(suppliers['supplier_id'], suppliers['reliability_score']))
DEFAULT_RELIABILITY = float(suppliers['reliability_score'].median())

# PCA-derived composite weights for [stockout_risk, overstock_flag, supplier
# reliability] — replaces a health score that only ever looked at stockout risk.
W_STOCKOUT, W_OVERSTOCK, W_RELIABILITY = get_inventory_health_weights()

print(f"Loaded stock information. Daily velocity calculated for {len(velocity)} products.")
print(f"Composite health weights (data-derived): stockout={W_STOCKOUT:.2f}, overstock={W_OVERSTOCK:.2f}, reliability={W_RELIABILITY:.2f}")

In [ ]:
def compute_inventory_health():
    # Merge stock details with velocity
    health_df = pd.merge(inv, velocity, on='product_id', how='left')
    health_df['daily_velocity'] = health_df['daily_velocity'].fillna(0.1) # default min
    
    # Calculate days cover
    health_df['days_cover'] = health_df['stock_qty'] / health_df['daily_velocity']
    
    # sigmoid risk: high risk when days_cover < reorder_point
    results = []
    for idx, row in health_df.iterrows():
        rop = row['reorder_point']
        cover = row['days_cover']
        
        # stockout risk function (sigmoid of difference)
        stockout_risk = 1.0 / (1.0 + np.exp((cover - rop) * 0.2))
        
        # Calculate overstock
        max_stk = row['max_stock'] if not pd.isna(row['max_stock']) else 9999.0
        overstock_flag = 1 if row['stock_qty'] > max_stk else 0

        # Real supplier reliability for this SKU's actual supplier
        supplier_id = product_supplier_map.get(row['product_id'])
        reliability = supplier_reliability_map.get(supplier_id, DEFAULT_RELIABILITY)

        # Genuine composite score: PCA-derived weights blending stockout risk,
        # overstock, and real supplier reliability (not stockout risk alone).
        health_score = float(np.clip(
            W_STOCKOUT * (1.0 - stockout_risk) +
            W_OVERSTOCK * (1.0 - overstock_flag) +
            W_RELIABILITY * reliability,
            0.0, 1.0
        ))
        
        label = "Healthy"
        if health_score < 0.4: label = "Critical"
        elif health_score < 0.7: label = "At Risk"
        
        results.append({
            "product_id": row['product_id'],
            "store_id": str(row['store_id']) if not pd.isna(row['store_id']) else "",
            "warehouse_id": str(row['warehouse_id']) if not pd.isna(row['warehouse_id']) else "",
            "location_type": str(row['location_type']) if not pd.isna(row['location_type']) else "",
            "stock_level": int(row['stock_qty']),
            "days_cover": round(float(cover), 1),
            "stockout_risk": round(float(stockout_risk), 3),
            "supplier_reliability": round(float(reliability), 3),
            "health_score": round(float(health_score), 2),
            "risk_label": label,
            "overstock_flag": overstock_flag
        })
    return results

inventory_health = compute_inventory_health()
print("Inventory Health metric:", json.dumps(inventory_health[0], indent=2))

In [ ]:
print("=== CALRETAIL INVENTORY CONTROL RADAR ===")
health_summary_df = pd.DataFrame(inventory_health)
# Group stats
print("SKU Classification Volume:")
print(health_summary_df['risk_label'].value_counts())
print("\nCritical Stock Items needing Replenishment:")
critical_items = health_summary_df[health_summary_df['risk_label'] == 'Critical'].sort_values('days_cover')
print(critical_items[['product_id', 'stock_level', 'days_cover', 'stockout_risk']].head(5).to_string(index=False))
